# Création des datasets du machine learning (train et test)

Ce notebook prépare les deux jeux de données utilisés par tous les notebooks du machine learning (`ml_0`, `ml_1`…) :
- `data/ML/train_dataset.csv` : le jeu d'**entraînement**, seul jeu lu pendant la mise au point des modèles (validation croisée) ;
- `data/ML/test_dataset.csv` : le jeu de **test**, qui n'est lu **qu'une seule fois**, à la toute fin du machine learning, une fois le modèle figé.

**Il ne dépend que du nettoyage** : il part du fichier du **niveau de nettoyage le plus élevé** produit par `src/02_01_nettoyage.ipynb` (`cleaned5_creditcard.csv` aujourd'hui), et non du dataset de Streamlit ni des notebooks d'EDA. Pour reproduire le ML, après un clone comme pour nous après toute correction de nettoyage : données brutes dans `data/raw/`, puis `02_01_nettoyage` exécuté en entier, puis ce notebook, puis les scénarios dans l'ordre (section 0 de `methodologie_ML.md`).

**Étapes** :
1. chargement du fichier du niveau de nettoyage le plus élevé, trié par `ID` ;
2. périmètre du ML : clients avec une dette en septembre (`BILL_AMT1 > 0`) et plafond de 500 000 NT$ ou moins ;
3. retrait des clients au contentieux à M, prédits en défaut par la règle métier (partie 5) ;
4. colonnes ajoutées pour le ML (calculées client par client), au moment où un scénario en a besoin ;
5. découpage 80/20, stratifié sur `dpnm`, `random_state=42`, sur les seuls clients laissés au ML ;
6. contrôles, puis enregistrement (avec la source des données, pour la traçabilité).


## 1. Chargement

On part du fichier du **niveau de nettoyage le plus élevé** présent dans `data/` (`cleanedN_creditcard.csv`, N le plus grand) : le nettoyage complet, codifications corrigées et colonnes du contentieux comprises (niveau 5). Si un nouveau niveau de nettoyage est créé, il est pris automatiquement.

Les clients sont **triés par `ID`** : le découpage ne dépend alors que des clients retenus et de leur `dpnm`, et plus de l'ordre des lignes du fichier.

⚠️ La date du fichier source est affichée pour information : après toute correction de nettoyage, relancer `02_01_nettoyage` en entier avant ce notebook.


In [1]:
import re
from datetime import datetime
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

def fr(n):
    # Nombre avec une espace pour séparer les milliers (23 456)
    return f"{n:,}".replace(",", " ")

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

# Fichier du niveau de nettoyage le plus élevé : data/cleanedN_creditcard.csv, N le plus grand
niveaux = {int(m.group(1)): f for f in (root_dir / "data").glob("cleaned*_creditcard.csv")
           if (m := re.fullmatch(r"cleaned(\d+)_creditcard\.csv", f.name))}
assert niveaux, "Aucun fichier de nettoyage dans data/ : exécuter d'abord src/02_01_nettoyage.ipynb"
NIVEAU_NETTOYAGE = max(niveaux)
fichier_source = niveaux[NIVEAU_NETTOYAGE]

df = pd.read_csv(fichier_source).sort_values("ID").reset_index(drop=True)
date_source = datetime.fromtimestamp(fichier_source.stat().st_mtime).strftime("%d/%m/%Y %H:%M")
print(f"Source            : {fichier_source.name} (niveau de nettoyage {NIVEAU_NETTOYAGE}, fichier du {date_source})")
print(f"Clients chargés   : {fr(len(df))}, triés par ID")
print(f"Colonnes          : {df.shape[1]}")


Source            : cleaned5_creditcard.csv (niveau de nettoyage 5, fichier du 05/10/2026 17:10)
Clients chargés   : 28 851, triés par ID
Colonnes          : 37


## 2. Le périmètre du machine learning

**Règle métier** : on cherche un risque réel. Un client sans dette en septembre (`BILL_AMT1 <= 0` : rien à rembourser, ou un avoir) n'a pas de crédit en cours sur lequel faire défaut ; il est retiré.
Les plafonds de plus de 500 000 NT$ sont déjà retirés par le nettoyage (niveau 4) : le filtre est gardé par sécurité, et la cellule vérifie qu'il ne retire personne.

In [2]:
nb_depart = len(df)
sans_dette = (df["BILL_AMT1"] <= 0).sum()
dont_avoir = (df["BILL_AMT1"] < 0).sum()
gros_plafonds = (df["LIMIT_BAL"] > 500000).sum()

s12 = df[(df["BILL_AMT1"] > 0) & (df["LIMIT_BAL"] <= 500000)].copy()

print(f"Clients au départ                        : {fr(nb_depart)}")
print(f"Retirés, sans dette en septembre         : {fr(sans_dette)} (dont {fr(dont_avoir)} avec un avoir)")
print(f"Retirés, plafond > 500 000 NT$           : {fr(gros_plafonds)} (attendu : 0, déjà retirés au niveau 4)")
print(f"Clients du périmètre                     : {fr(len(s12))}")

Clients au départ                        : 28 851
Retirés, sans dette en septembre         : 1 649 (dont 507 avec un avoir)
Retirés, plafond > 500 000 NT$           : 0 (attendu : 0, déjà retirés au niveau 4)
Clients du périmètre                     : 27 202


## 3. Retrait des clients au contentieux à M

Un client est au contentieux à M si `FLAG_CTX` = 1 et `MOIS_SORTIE_CTX` = -1 (définition : `docs/colonnes_creees.md`, page 5.3). Ces clients sont prédits en défaut par la règle métier, sans modèle : ils sont retirés avant le découpage, et le machine learning ne les voit pas.
Les clients sortis du contentieux, ou qui ont eu un retard isolé, restent : leur historique est résumé par les colonnes du contentieux (`FLAG_CTX`, `MOIS_SORTIE_CTX`, `FLAG_RETARD`, `MOIS_SORTIE_RETARD`, `NB_MOIS_CTX`…).

In [3]:
ctx_a_m = (s12["FLAG_CTX"] == 1) & (s12["MOIS_SORTIE_CTX"] == -1)
population_ml = s12[~ctx_a_m].copy()

resume = pd.DataFrame({
    "Clients": [len(s12), ctx_a_m.sum(), len(population_ml)],
    "Taux de défaut (%)": [s12["dpnm"].mean() * 100, s12.loc[ctx_a_m, "dpnm"].mean() * 100, population_ml["dpnm"].mean() * 100],
}, index=["Périmètre", "Contentieux à M (retirés)", "Clients laissés au ML"]).round(2)
display(resume)

,Clients,Taux de défaut (%)
Périmètre,27202,21.95
Contentieux à M (retirés),3004,70.47
Clients laissés au ML,24198,15.93


**Lecture** : une ligne par population. Les clients au contentieux à M font très souvent défaut ; une fois retirés, le taux de défaut des clients laissés au ML baisse nettement. Les modèles travaillent sur une population plus homogène, mais plus difficile : les défauts les plus visibles sont déjà traités par la règle.

## 4. Colonnes ajoutées pour le machine learning

**Règle** : une colonne ajoutée ici se calcule **client par client**, à partir des seules données de ce client et de seuils fixés à l'avance par la logique métier (jamais en regardant `dpnm`). Elle ne dépend ni des autres clients ni de la cible : elle peut donc être calculée avant le découpage, et le test reçoit exactement le même calcul que le train.
Tout ce qui s'apprend sur l'ensemble des clients (normalisation, encodage, remplissage par une moyenne, tranches par quantiles) se fait dans le pipeline des notebooks de ML, pas ici.

**Une colonne n'est ajoutée qu'au moment où un scénario en a besoin**, d'après sa définition dans `docs/colonnes_creees.md` (seules les colonnes répertoriées entrent dans le ML ; un contrôle de l'étape 6 bloque l'enregistrement sinon). Pour l'instant, aucune colonne n'est ajoutée : le socle `ml_0` n'utilise que les variables d'origine.

**Une seule définition, deux codes** : certaines colonnes sont aussi calculées par l'EDA (dataset de Streamlit). Le contrôle de l'étape 6 compare chaque colonne présente des deux côtés, client par client : un écart bloque l'enregistrement, pour que les deux calculs ne divergent jamais.


In [4]:
# Colonnes ajoutées pour le ML (calcul client par client, d'après docs/colonnes_creees.md), au moment où un scénario en a besoin
# Exemple : population_ml["NOUVELLE_COLONNE"] = ...

print(f"Colonnes disponibles pour le ML ({population_ml.shape[1]}) :")
print(", ".join(population_ml.columns))


Colonnes disponibles pour le ML (37) :
ID, LIMIT_BAL, SEX, EDUCATION, MARRIAGE, AGE, PAY_1, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6, BILL_AMT1, BILL_AMT2, BILL_AMT3, BILL_AMT4, BILL_AMT5, BILL_AMT6, PAY_AMT1, PAY_AMT2, PAY_AMT3, PAY_AMT4, PAY_AMT5, PAY_AMT6, dpnm, ratio_PAY_BILL1, ratio_PAY_BILL2, ratio_PAY_BILL3, ratio_PAY_BILL4, ratio_PAY_BILL5, FAUX_CODAGE, SURVEILLANCE_RECENTE, FLAG_CTX, MOIS_SORTIE_CTX, FLAG_RETARD, MOIS_SORTIE_RETARD, NB_MOIS_CTX


## 5. Découpage train / test

Découpage 80/20 des clients laissés au ML, stratifié sur `dpnm` (même part de défauts dans les deux jeux), `random_state=42` : le tirage est reproductible.

In [5]:
train, test = train_test_split(population_ml, test_size=0.2, stratify=population_ml["dpnm"], random_state=42)

decoupage = pd.DataFrame({
    "Clients": [len(train), len(test)],
    "Défauts": [train["dpnm"].sum(), test["dpnm"].sum()],
    "Taux de défaut (%)": [train["dpnm"].mean() * 100, test["dpnm"].mean() * 100],
}, index=["Train", "Test"]).round(2)
display(decoupage)

,Clients,Défauts,Taux de défaut (%)
Train,19358,3084,15.93
Test,4840,771,15.93


**Lecture** : une ligne par jeu. Le taux de défaut est le même dans les deux jeux, à l'arrondi près : c'est l'effet du découpage stratifié.

## 6. Contrôles

1. **Le découpage ne dépend pas des colonnes** : on refait le découpage avec les seules colonnes `ID` et `dpnm`, et on vérifie que le test contient exactement les mêmes clients. Ajouter une colonne à l'étape 4 ne change donc pas la répartition des clients.
2. **Aucun client en double, aucun client perdu** entre la population du ML, le train et le test.
3. **Aucun client au contentieux à M** dans les jeux enregistrés.
4. **Toutes les colonnes sont connues** : chaque colonne est soit une des 25 colonnes d'origine (identifiant, cible et 23 variables), soit une colonne répertoriée dans `docs/colonnes_creees.md` (les noms en `n`, comme `ratio_PAY_BILLn`, valent pour leurs numéros de mois). Sinon, le notebook s'arrête avant l'enregistrement.
5. **Mêmes valeurs que le dataset de Streamlit** pour chaque colonne présente des deux côtés, client par client (à l'arrondi de stockage près). Sinon, le notebook s'arrête : un même calcul ne doit jamais donner deux résultats.


In [6]:
_, test_ids = train_test_split(population_ml[["ID", "dpnm"]], test_size=0.2, stratify=population_ml["dpnm"], random_state=42)

def nb_ctx_a_m(d):
    return ((d["FLAG_CTX"] == 1) & (d["MOIS_SORTIE_CTX"] == -1)).sum()

print("Même test sans les autres colonnes        :", set(test["ID"]) == set(test_ids["ID"]))
print("Aucun client commun au train et au test   :", len(set(train["ID"]) & set(test["ID"])) == 0)
print("Train + test = clients laissés au ML      :", len(train) + len(test) == len(population_ml))
print("Aucun contentieux à M (train, test)       :", nb_ctx_a_m(train) == 0 and nb_ctx_a_m(test) == 0)

# Colonnes d'origine et colonnes répertoriées dans docs/colonnes_creees.md
import re
colonnes_origine = ["ID", "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE", "dpnm"] + [
    f"{prefixe}{n}" for prefixe in ("PAY_", "BILL_AMT", "PAY_AMT") for n in range(1, 7)]
doc = (root_dir / "docs" / "colonnes_creees.md").read_text(encoding="utf-8")
noms_doc = set(re.findall(r"`([A-Za-z_][A-Za-z_0-9]*)`", doc))
familles = [nom[:-1] for nom in noms_doc if nom.endswith("n")]   # ratio_PAY_BILLn -> ratio_PAY_BILL1 à 5

def repertoriee(col):
    return col in noms_doc or any(re.fullmatch(re.escape(f) + r"\d", col) for f in familles)

inconnues = [c for c in train.columns if c not in colonnes_origine and not repertoriee(c)]
print("Colonnes non répertoriées                 :", inconnues if inconnues else "aucune")
assert not inconnues, f"Colonnes à définir dans docs/colonnes_creees.md avant d'entrer dans le ML : {inconnues}"
# Contrôle 5 : mêmes valeurs que le dataset de Streamlit, pour chaque colonne présente des deux côtés (à l'arrondi de stockage près)
streamlit = pd.read_csv(root_dir / "data" / "csv_streamlit" / "dataset_streamlit.csv").set_index("ID")
communes = [c for c in population_ml.columns if c != "ID" and c in streamlit.columns]
ecarts = {}
for col in communes:
    a = population_ml.set_index("ID")[col]
    b = streamlit.loc[a.index, col]
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        egal = np.isclose(a, b, rtol=1e-9, atol=1e-9, equal_nan=True)
    else:
        egal = (a == b) | (a.isna() & b.isna())
    if not np.all(egal):
        ecarts[col] = int((~np.asarray(egal)).sum())
print(f"Colonnes comparées au dataset de Streamlit : {len(communes)} ; écarts :", ecarts if ecarts else "aucun")
assert not ecarts, f"Colonnes qui diffèrent du dataset de Streamlit (nombre de clients) : {ecarts}"


Même test sans les autres colonnes        : True
Aucun client commun au train et au test   : True
Train + test = clients laissés au ML      : True
Aucun contentieux à M (train, test)       : True
Colonnes non répertoriées                 : aucune
Colonnes comparées au dataset de Streamlit : 36 ; écarts : aucun


## 7. Enregistrement

Les deux fichiers sont enregistrés dans `data/ML/` (ignorés par git, comme les autres CSV du dossier `data/`). L'index d'origine n'est pas gardé : l'identifiant du client est la colonne `ID`.
La **source des données** est enregistrée à côté, dans `data/ML/source_datasets.json` (fichier et niveau de nettoyage, date de création) : les notebooks de ML la recopient dans le fichier de résultats, pour savoir sur quelles données chaque résultat a été calculé.
**Rappel** : les notebooks de mise au point ne lisent que `train_dataset.csv`.


In [7]:
dossier_ml = root_dir / "data" / "ML"
dossier_ml.mkdir(parents=True, exist_ok=True)

train.to_csv(dossier_ml / "train_dataset.csv", index=False)
test.to_csv(dossier_ml / "test_dataset.csv", index=False)

print(f"Enregistré : {dossier_ml / 'train_dataset.csv'} ({fr(len(train))} clients)")
print(f"Enregistré : {dossier_ml / 'test_dataset.csv'} ({fr(len(test))} clients)")
import json
source = {"fichier_source": fichier_source.name, "niveau_nettoyage": NIVEAU_NETTOYAGE, "date_fichier_source": date_source,
          "date_creation": datetime.now().strftime("%d/%m/%Y %H:%M"), "clients_train": len(train), "clients_test": len(test)}
(dossier_ml / "source_datasets.json").write_text(json.dumps(source, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Enregistré : {dossier_ml / 'source_datasets.json'} {source}")


Enregistré : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\data\ML\train_dataset.csv (19 358 clients)
Enregistré : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\data\ML\test_dataset.csv (4 840 clients)
Enregistré : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\data\ML\source_datasets.json {'fichier_source': 'cleaned5_creditcard.csv', 'niveau_nettoyage': 5, 'date_fichier_source': '05/10/2026 17:10', 'date_creation': '05/10/2026 17:10', 'clients_train': 19358, 'clients_test': 4840}
